# TEM Specimen Tilt Navigation

You are at the microscope. You have chased Kikuchi bands to *some* zone axis, indexed
the pattern, and the stage reads some $(\alpha, \beta)$. Now you want a **different**
zone axis. Which way do you tilt? How far? Can the holder even get there? And when you
arrive, which of the symmetry-equivalent versions of your target are you actually
looking down?

This notebook answers all of that on a worked case, and — more importantly — shows the
two things that make the problem harder than it looks:

1. **The geometry is the easy half.** The holder angles come out of two lines of
   `atan2`. Almost everything in `pytex.tem` is about enumerating candidates and telling
   *choices* apart from *hypotheses*, not about solving equations.
2. **The failure that costs you a session is not the one you have been warned about.**
   The famous "180 degree ambiguity" of SAED indexing is, for any cubic or hexagonal
   metal, completely harmless. The error that actually wastes an afternoon is an
   uncalibrated diffraction rotation — and it reports a clean zero residual while
   driving the specimen in exactly the wrong direction.

Everything below is computed live. See
{doc}`../../architecture/tem_tilt_navigation_foundation` for the derivations.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore", category=UserWarning, module="pymatgen.*")

from pytex import (
    CurrentState,
    DoubleTiltStage,
    EllipticalEnvelope,
    FrameDomain,
    Handedness,
    Lattice,
    Orientation,
    Phase,
    RectangularEnvelope,
    ReferenceFrame,
    StageCalibration,
    StagePosition,
    SymmetrySpec,
    ZoneAxis,
    ZoneAxisObservation,
    analyze_ambiguity,
    plan_tilt_to_zone_axis,
    solve_tilts_for_direction,
)
from pytex.plotting.tilt_stereogram import plot_tilt_stereogram
from pytex.tem.calibration import (
    TiltExcursionObservation,
    calibrate_from_tilt_excursions,
    predicted_excursion_azimuth_deg,
    residual_from_rotation_error_deg,
)
from pytex.tem.reconstruction import HOLDER_FRAME

plt.rcParams["figure.dpi"] = 110

## 1. The specimen and the holder

Nickel, FCC, point group $m\bar{3}m$ — the ordinary case, and deliberately so: the whole
point of section 6 is that the ambiguity everybody worries about does *not* arise here.

The holder is a conventional double-tilt cartridge with **asymmetric** limits. Note the
elliptical envelope offered alongside it, in which each range shrinks as the other grows.
Real cartridges behave that way, and modelling an elliptical envelope as a rectangle is
how a plan ends up asking for a position the stage cannot reach.

In [ ]:
crystal = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"), Handedness.RIGHT)

nickel = Phase(
    "nickel-fcc",
    lattice=Lattice(3.52387, 3.52387, 3.52387, 90.0, 90.0, 90.0, crystal_frame=crystal),
    symmetry=SymmetrySpec.from_point_group("m-3m", reference_frame=crystal),
    crystal_frame=crystal,
    space_group_symbol="Fm-3m",
)

holder = DoubleTiltStage(
    name="double-tilt cartridge",
    envelope=RectangularEnvelope(-35.0, 35.0, -25.0, 30.0),
    calibration=StageCalibration(backlash_deg=0.3, angular_uncertainty_deg=0.15),
)
print(holder.describe())

The last sentences of that description are the ones to read. **The diffraction rotation
is not calibrated** — and the engine says so, and names the reconstruction path that does
not need it. That is deliberate: an uncalibrated instrument is the normal state of
affairs, and pretending otherwise is what produces confident wrong answers.

Notice also the reachable solid angle. A double-tilt holder commands under ten percent of
all beam directions. That one number is why symmetry equivalents matter so much in
practice, and it follows from an elementary integral: the beam direction in holder
coordinates is

$$\hat{\mathbf{b}}_H(\alpha,\beta) = \left(-\cos\alpha\,\sin\beta,\ \sin\alpha,\ \cos\alpha\,\cos\beta\right),$$

which is a spherical coordinate system whose **pole is the $\beta$ axis**. Its Jacobian is
$\cos\alpha$, so $\Omega = \Delta\beta\,(\sin\alpha_{\max} - \sin\alpha_{\min})$.

In [ ]:
for name, envelope in [
    ("+/-30 / +/-30 rectangle", RectangularEnvelope(-30.0, 30.0, -30.0, 30.0)),
    ("+/-40 / +/-30 rectangle", RectangularEnvelope(-40.0, 40.0, -30.0, 30.0)),
    ("this cartridge", holder.envelope),
    ("elliptical 30 / 30", EllipticalEnvelope(30.0, 30.0)),
]:
    omega = envelope.accessible_solid_angle_sr()
    print(f"{name:26}  {omega:6.3f} sr   {100 * omega / (4 * np.pi):5.2f}% of all directions")

The elliptical envelope reaches noticeably less than the rectangle with the same nominal
limits, because its corners are missing — exactly the corners a naive plan would try to
use.

## 2. Where is the crystal? Two indexed zone axes are enough

Here is the result that reorganises the whole design.

To compute a tilt you need the crystal-to-holder orientation $\mathbf{U}$. The obvious
way to get it is from one indexed pattern plus the calibrated diffraction rotation
$\varphi_D$ — and $\varphi_D$ is the single hardest constant in the problem, is not in the
file metadata, and drifts with the lens history.

But if you have indexed **two** zone axes at two stage positions, then by the master
equation $\mathbf{U}\hat{\mathbf{n}}_i = \hat{\mathbf{b}}_H(\alpha_i, \beta_i)$, and two
non-parallel vector correspondences determine a rotation uniquely. **No pattern rotation,
no parity bit, no detector model.** Since anyone who chased Kikuchi bands to get here has
almost certainly seen a second zone, this is the path to use.

Let us simulate an operator who found $[001]$ and then $[\bar{1}13]$.

In [ ]:
# Ground truth, used only to generate the synthetic observations. The engine never sees it.
truth = Orientation.from_matrix(
    np.array(
        [
            [0.86217916, -0.29883624, 0.40886634],
            [0.42262661, 0.87491009, -0.23180177],
            [-0.27829338, 0.38139448, 0.88164556],
        ]
    ),
    specimen_frame=HOLDER_FRAME,
    phase=nickel,
    crystal_frame=crystal,
)

first_zone = ZoneAxis([0, 0, 1], phase=nickel)
second_zone = ZoneAxis([-1, 1, 3], phase=nickel)

# Where the stage must sit for each zone to be on the beam, under the truth.
first_position = StagePosition(
    *solve_tilts_for_direction(truth.as_matrix() @ first_zone.unit_vector)[0]
)
second_position = StagePosition(
    *solve_tilts_for_direction(truth.as_matrix() @ second_zone.unit_vector)[0]
)
print("operator found [001]  at", first_position)
print("operator found [-113] at", second_position)

In [ ]:
current = CurrentState.from_two_zone_axes(
    ZoneAxisObservation(first_zone, first_position, label="first pattern"),
    ZoneAxisObservation(second_zone, second_position, label="second pattern"),
    holder,
)
print(current.describe())

Two things in that output earn their place.

**The consistency residual.** The angle between two zone axes is a crystallographic
invariant, and the angle between the two beam directions follows from the stage model
alone. They must agree. A residual of a few tenths of a degree is indexing error; tens of
degrees indicts a reversed sign convention, a mis-indexed zone, or a bent specimen — and
the check costs nothing, needs no calibration, and catches the errors that matter.

**The verdict on the residual two-fold.** Flipping *both* zone-axis senses admits a second
valid rotation, related by a $180^\circ$ turn about $\hat{\mathbf{n}}_1 \times
\hat{\mathbf{n}}_2$. The engine checks whether that two-fold is itself a crystal symmetry
and, if it is, says so — no warning, no experiment needed.

Let us confirm the reconstruction actually recovered the truth, modulo crystal symmetry.

In [ ]:
operators = np.asarray(nickel.symmetry.operators)
relative = np.einsum(
    "ij,njk->nik", current.matrix, np.einsum("nij,jk->nik", operators, truth.as_matrix().T)
)
angles = np.degrees(
    np.arccos(np.clip((np.trace(relative, axis1=1, axis2=2) - 1.0) / 2.0, -1.0, 1.0))
)
print(f"smallest symmetry-reduced angle to the truth: {angles.min():.3e} deg")

## 3. Tilting to a target

Now the actual question. We are down $[\bar{1}13]$ and we want $\langle 011 \rangle$ — a
common move, since $\langle 011 \rangle$ in an FCC metal gives the pattern with the
strongest $\{111\}$ reflections for dislocation work.

In [ ]:
report = plan_tilt_to_zone_axis(current, ZoneAxis([0, 1, 1], phase=nickel), holder)
print(report.describe())

Read the numbers rather than the prose. The orbit of $\langle 011 \rangle$ contains twelve
distinct directions, and only a fraction fall inside the holder. **The user asked for one
direction; the crystal offered twelve, and the engine took the cheapest reachable one.**
Every member gives an identical diffraction pattern, so that substitution is free — which
is precisely why a target that looks unreachable so often is not.

The plan also names the **Kikuchi band to follow**. That is not decoration: the geodesic
between two zone axes lies in the plane they span, and the band of the reflection whose
normal is that plane's normal is exactly the band joining the two poles. The
mathematically optimal path is the one experienced operators already follow by eye.

In [ ]:
best = report.best()
print(f"drive to            alpha {best.position.alpha_deg:+.2f} deg, beta {best.position.beta_deg:+.2f} deg")
print(f"change of           alpha {best.delta_alpha_deg:+.2f} deg, beta {best.delta_beta_deg:+.2f} deg")
print(f"lands on            {best.orbit_member_indices}")
print(f"forward residual    {best.residual_deg:.2e} deg")
print(f"specimen rotation   {best.travel_deg:.2f} deg")
print(f"crystal travel      {best.path.total_travel_deg:.2f} deg")
print(f"envelope clearance  {best.envelope_margin_deg:.2f} deg")
print(f"beta conditioning   {best.conditioning:.3f}  (1/cos alpha at the destination)")
print(f"path stays inside   {best.path.is_valid}")

Note that **specimen rotation and crystal travel are different numbers**. The specimen
rotation is the angle of the actual rigid-body rotation $\mathbf{R}_{\text{target}}
\mathbf{R}_{\text{current}}^{\mathsf{T}}$; the crystal travel is the arc the beam
direction sweeps across the crystal, which is what a Kikuchi pattern moves through. The
naive $\sqrt{\Delta\alpha^2 + \Delta\beta^2}$ is neither, and is not the angle of
anything.

## 4. The figure

One stereogram, two panels. The overview places the move among the low-index poles; the
detail zooms to the reachable region, where the stage angles and the sense of each knob
can be read.

The trajectory is drawn as **dots, one per sampled stage position**. Their spacing carries
information — where they crowd, the beam is moving slowly through the crystal for a given
change of stage angle — and they grow toward the target, so the direction of travel
survives a greyscale reprint.

In [ ]:
figure = plot_tilt_stereogram(report, holder)
plt.show()

Everything on that figure comes from numbers the engine produced. The renderer does no
kinematics of its own: it plots the `TiltPath` samples. That matters, because a drawing
which agreed with the engine by re-implementing the same formula would be evidence about
nothing.

Three things to look for:

- The **blue outline** is the set of crystal directions this holder can put on the beam.
  It draws as exact circular arcs, not a sampled blob, because constant-$\beta$ curves are
  great circles through the $\beta$ pole and constant-$\alpha$ curves are small circles
  about it.
- The **red rings** are symmetry equivalents of the target that fall outside the envelope;
  the **green ring** marks one that does not. Most of the orbit is out of reach. One member
  is enough.
- The **amber arcs** show which way the beam moves in the crystal when each knob is turned
  positive. This is the answer to "what does $+\alpha$ actually do to my pattern", drawn
  from the calibrated forward model rather than from an assumption.

## 5. When the target really is out of reach

Ask for something that is not there, and the engine must not simply fail.

In [ ]:
hard = plan_tilt_to_zone_axis(current, ZoneAxis([1, 1, 1], phase=nickel), holder)
print("reachable:", hard.is_reachable)
print()
print(hard.describe())

In [ ]:
nearest = hard.nearest_approach
if nearest is not None:
    print(f"nearest approach at  {nearest.position}")
    print(f"leaves the target    {nearest.residual_deg:.2f} deg off the beam")
    print(f"inside the envelope? {holder.envelope.contains(*nearest.position.as_tuple())}")

A nearest approach is a real answer, not a consolation. A miss of a few degrees still puts
the Kikuchi pole in view, and the operator can judge whether that is enough. What the API
refuses to do is let a nearest approach be mistaken for a hit: the four verdicts —
`EXACT`, `WITHIN_TOLERANCE`, `NEAREST_APPROACH`, `UNREACHABLE` — are qualitatively
different answers and are kept apart.

Note too that the nearest approach is **guaranteed to be a position the holder can
actually reach**. An exact solution sitting outside the envelope is not an approach to
anything.

## 6. The ambiguity that does not matter, and the one that does

### 6.1 Friedel's law: harmless here, and the engine says so

A single kinematic SAED pattern is centrosymmetric even when the crystal is not, because
$|F(\mathbf{g})| = |F(-\mathbf{g})|$. So the orientation is determined only up to the
rotations of the **Laue class** that map the zone plane to itself.

The decisive question is whether those rotations are ones the crystal already has.

In [ ]:
for label, indices in [("[001]", [0, 0, 1]), ("[011]", [0, 1, 1]), ("[111]", [1, 1, 1])]:
    zone = ZoneAxis(indices, phase=nickel)
    ambiguity = analyze_ambiguity(nickel, zone.unit_vector)
    print(
        f"nickel down {label:6} stabilizer order {ambiguity.stabilizer_order:2}, "
        f"of which {ambiguity.symmetry_stabilizer_order:2} are crystal symmetries "
        f"-> {'UNIQUE' if ambiguity.is_unique else 'AMBIGUOUS'}"
    )

Every operator that could confuse the indexing is already a crystal symmetry of
$m\bar{3}m$. **For a centrosymmetric crystal, Friedel's law adds nothing.** The much-feared
$180^\circ$ ambiguity is entirely absorbed by symmetry, and the engine reports no
ambiguity rather than warning about one.

Enumerating all 32 point groups, the Laue rotation group is strictly larger — by a factor
of exactly two — for only **ten** of them: those containing improper operations other than
inversion. Note that "non-centrosymmetric" is the *wrong* test: quartz, point group 32, is
non-centrosymmetric and yet completely ambiguity-free, because its point group is already
all rotations.

In [ ]:
from pytex.core.point_groups import PointGroup, all_point_group_symbols
from pytex.tem.ambiguity import AMBIGUOUS_POINT_GROUPS, laue_rotation_operators

print(f"{'group':>7} {'|proper|':>9} {'|Laue rot|':>11} {'factor':>7}")
for symbol in all_point_group_symbols():
    proper = int(np.sum(np.linalg.det(PointGroup.from_symbol(symbol).operators) > 0))
    laue = len(laue_rotation_operators(symbol))
    factor = laue // proper
    flag = "  <-- Friedel adds a rotation" if factor > 1 else ""
    print(f"{symbol:>7} {proper:9} {laue:11} {factor:7}{flag}")
print()
print("affected groups:", ", ".join(sorted(AMBIGUOUS_POINT_GROUPS)))
print("count:", len(AMBIGUOUS_POINT_GROUPS), "of 32")

Gallium arsenide ($\bar{4}3m$) is one of the ten. Down $[001]$ it genuinely does have two
competing orientation hypotheses — and there the engine must, and does, refuse to pick one
silently.

In [ ]:
gaas = Phase(
    "gallium-arsenide",
    lattice=Lattice(5.6533, 5.6533, 5.6533, 90.0, 90.0, 90.0, crystal_frame=crystal),
    symmetry=SymmetrySpec.from_point_group("-43m", reference_frame=crystal),
    crystal_frame=crystal,
)
print(analyze_ambiguity(gaas, [0.0, 0.0, 1.0]).describe())

Two families, each emitted with its own tilts, and a ranked list of experiments that
discriminate them — cheapest decisive test first. Note that the recommended test is not
CBED: it is a one-minute tilt excursion.

### 6.2 The error that actually costs you a session

Now the one nobody warns you about. Suppose the diffraction rotation $\varphi_D$ is wrong.
That is a rotation about the beam axis. It is **not** absorbed by crystal symmetry, and the
resulting miss is

$$\Delta = 2\arcsin\!\left(\sin\tfrac{\delta\varphi}{2}\,\sin\theta\right) \approx \delta\varphi\,\sin\theta,$$

where $\theta$ is the angle between your current and target zone axes.

In [ ]:
print(f"{'error':>8}   {'5 deg hop':>10} {'30 deg hop':>11} {'60 deg hop':>11} {'90 deg hop':>11}")
for error in (2.0, 5.0, 15.0, 180.0):
    misses = [residual_from_rotation_error_deg(error, hop) for hop in (5.0, 30.0, 60.0, 90.0)]
    print(f"{error:6.0f} deg   " + " ".join(f"{m:10.2f}" for m in misses))

Two things fall straight out of that table.

**The error grows with the length of the hop.** A $5^\circ$ calibration error is negligible
over a short move and fatal over a long one. That is the argument for routing a long
excursion through intermediate low-index zones and re-indexing at each: several short hops
are self-correcting where one long hop is open-loop. The engine suggests waypoints
automatically when a hop exceeds $30^\circ$.

**And at $\delta\varphi = 180^\circ$ it is a catastrophe.** From a starting position of
zero tilt the error negates both angles exactly, sending the specimen in precisely the
wrong direction. From an arbitrary starting position — as below — the answer is instead a
completely different pair of angles, landing on a different symmetry equivalent
altogether. Either way **the calculation reports a clean zero residual**, because it is
faithfully solving the geometry it was handed. Nothing about the arithmetic looks wrong.

In [ ]:
from pytex.tem.stage import rotation_z

def state_with_rotation_error(error_deg):
    wrong = rotation_z(np.deg2rad(error_deg)) @ truth.as_matrix()
    return CurrentState.from_orientation(
        Orientation.from_matrix(
            wrong, specimen_frame=HOLDER_FRAME, phase=nickel, crystal_frame=crystal
        ),
        current.position,
        current_zone_axis=second_zone,
    )

target = ZoneAxis([0, 1, 1], phase=nickel)
good = plan_tilt_to_zone_axis(current, target, holder, include_paths=False).best()
bad = plan_tilt_to_zone_axis(
    state_with_rotation_error(180.0), target, holder, include_paths=False
).best()

print(f"correct calibration:  alpha {good.position.alpha_deg:+7.2f}, beta {good.position.beta_deg:+7.2f}   residual {good.residual_deg:.2e} deg")
print(f"180 deg wrong:        alpha {bad.position.alpha_deg:+7.2f}, beta {bad.position.beta_deg:+7.2f}   residual {bad.residual_deg:.2e} deg")
print()
print("Both report a residual of zero. Only one puts the crystal where you want it.")
print("The residual measures how well the solver did, not whether the inputs were true.")

## 7. Calibrating it away in two exposures

The remedy is cheap, and it is the most valuable minute you will spend at the microscope.

At a position where a Kikuchi pattern is visible: record a reference, apply a known
positive $\alpha$ of 5–10 degrees, record where a tracked feature moved to, return, and
repeat for $\beta$.

A rigid crystal rotation carries a Kikuchi pole at the pattern centre along
$-\hat{\mathbf{y}}_{\text{lab}}$ for positive $\alpha$ and along
$+\hat{\mathbf{x}}_{\text{lab}}$ for positive $\beta$. Comparing the measured azimuths
against those predictions gives $\varphi_D = -\psi_\beta$ directly, and the **sign** of
$\psi_\alpha - \psi_\beta$ gives the parity of the stored image. Three results from two
exposures, plus a built-in self-check: the two azimuths must be $90^\circ$ apart.

In [ ]:
# Simulate an instrument whose true diffraction rotation is +37 degrees and whose camera
# writes a mirrored image.
TRUE_ROTATION, TRUE_MIRROR = 37.0, True

measured = calibrate_from_tilt_excursions(
    TiltExcursionObservation(
        "alpha",
        8.0,
        predicted_excursion_azimuth_deg("alpha", TRUE_ROTATION, mirrored=TRUE_MIRROR),
        displacement_deg=8.0,
        feature="(220) Kikuchi pole",
    ),
    TiltExcursionObservation(
        "beta",
        8.0,
        predicted_excursion_azimuth_deg("beta", TRUE_ROTATION, mirrored=TRUE_MIRROR),
        displacement_deg=8.0,
        feature="(220) Kikuchi pole",
    ),
    camera_length_mm=800.0,
    accelerating_voltage_kv=200.0,
)
print(measured.describe())
print()
print(f"true rotation {TRUE_ROTATION:+.1f} deg, mirrored={TRUE_MIRROR}")

Recovered exactly, parity included. Note the final sentence: the calibration records the
camera length it was measured at and **refuses to be applied at another one**, because the
diffraction rotation is hysteretic in the projector and diffraction lens settings.
Interpolating a hysteretic quantity manufactures a plausible wrong number, which is
precisely the failure this procedure exists to prevent.

In [ ]:
calibrated = measured.calibration
calibrated.check_applicable(camera_length_mm=800.0)  # fine
try:
    calibrated.check_applicable(camera_length_mm=1200.0)
except ValueError as error:
    print("refused:", error)

## 8. The reverse direction: orientation *out* of indexing

Everything so far inverts the master equation for the **tilts**. The same equation inverts
for the **orientation**, which is what texture and microstructure work actually wants out of
an indexed pattern:

$$\mathbf{U} = \mathbf{R}_{\text{stage}}(\alpha,\beta)^{\mathsf{T}}\ \mathbf{F}\ \mathbf{R}_z(\varphi_D)\ \mathbf{R}_{P\leftarrow C}.$$

Because the holder frame *is* the specimen frame, the result is an ordinary `Orientation` —
reportable in Bunge $(\varphi_1, \Phi, \varphi_2)$ and directly comparable with an EBSD
measurement, with no convention conversion in between.

Let us simulate what indexing would have reported for our known crystal, then feed it back
in and see whether we recover the orientation we started from.

In [ ]:
from pytex import (
    IndexedPatternObservation,
    orientation_from_indexed_pattern,
    orientation_from_indexed_patterns,
)
from pytex.tem.stage import rotation_z

TRUE_PHI_D = 37.0  # the instrument's diffraction rotation, unknown to the operator

def indexing_would_report(orientation, position, phi_d):
    """Invert the composition to get the crystal-to-pattern rotation."""
    stage_matrix = holder.rotation_matrix(position.alpha_deg, position.beta_deg)
    return rotation_z(np.deg2rad(-phi_d)) @ stage_matrix @ orientation

def position_of(orientation, zone):
    from pytex import solve_tilts_for_direction
    return StagePosition(*solve_tilts_for_direction(orientation @ zone.unit_vector)[0])

### 8.1 One pattern, with a calibration

If the diffraction rotation is known, one pattern and the stage readings are enough.

In [ ]:
calibrated_holder = DoubleTiltStage(
    envelope=holder.envelope,
    calibration=StageCalibration(diffraction_rotation_deg=TRUE_PHI_D),
)

zone = ZoneAxis([0, 0, 1], phase=nickel)
where = position_of(truth.as_matrix(), zone)
reported = indexing_would_report(truth.as_matrix(), where, TRUE_PHI_D)

indexed = orientation_from_indexed_pattern(reported, zone, where, calibrated_holder)
print(indexed.describe())
print()
print("Bunge (phi1, Phi, phi2) =", tuple(round(a, 4) for a in indexed.euler_bunge_deg))
print("max |recovered - truth| =", float(np.max(np.abs(indexed.matrix - truth.as_matrix()))))

Recovered to machine precision. But note what that depended on: **we told it the diffraction
rotation**. Without one it refuses outright rather than guessing, because a guessed value
rotates the reported orientation bodily about the beam axis and crystal symmetry does not
absorb that error.

In [ ]:
try:
    orientation_from_indexed_pattern(reported, zone, where, holder)  # holder: uncalibrated
except ValueError as error:
    print("refused:", str(error)[:260], "...")

### 8.2 Two patterns, and no calibration at all

Here is the part worth the most. With **two** indexed patterns at different stage positions,
the orientation *and* the diffraction rotation come out together, from the data alone.

The zone axes fix the orientation by themselves. Each pattern's in-plane indexing then
over-determines $\varphi_D$: the residual
$\mathbf{M}_i = \mathbf{R}_{\text{stage},i}\mathbf{U}\mathbf{R}_i^{\mathsf{T}}$ must be a
rotation **about the beam axis**, and its angle *is* the diffraction rotation — read off, not
searched for.

In [ ]:
observations = []
for indices in ([0, 0, 1], [0, 1, 1], [-1, 1, 3]):
    z = ZoneAxis(indices, phase=nickel)
    p = position_of(truth.as_matrix(), z)
    observations.append(
        IndexedPatternObservation(
            indexing_would_report(truth.as_matrix(), p, TRUE_PHI_D), z, p, label=str(indices)
        )
    )

# `holder` carries NO diffraction-rotation calibration.
fit = orientation_from_indexed_patterns(observations, holder)
print(fit.describe())

In [ ]:
ops = np.asarray(nickel.symmetry.operators)
rel = np.einsum("ij,njk->nik", fit.matrix, np.einsum("nij,jk->nik", ops, truth.as_matrix().T))
err = np.degrees(np.arccos(np.clip((np.trace(rel, axis1=1, axis2=2) - 1.0) / 2.0, -1.0, 1.0))).min()
print(f"planted diffraction rotation : {TRUE_PHI_D:+.4f} deg")
print(f"recovered                    : {fit.diffraction_rotation_deg:+.4f} deg")
print(f"orientation error vs truth   : {err:.3e} deg (symmetry-reduced)")

Both recovered exactly, from patterns alone. That closes the loop: run this once at the start
of a session and the resulting calibration converts every later single pattern directly.

In [ ]:
session_calibration = fit.as_calibration()
print(session_calibration.describe())

### 8.3 The diagnostic that matters

Two independent checks come out of the fit, and the second is the one to read.

The **scatter** of $\varphi_D$ across patterns says whether they agree on one instrument
constant. The **beam-axis deviation** says something stronger: it is the part of the residual
that *no* value of $\varphi_D$ could absorb, because it is orthogonal to that entire degree of
freedom. A non-zero value means an input is wrong — a mirrored stored pattern, a mis-indexed
reflection, a reversed stage sign, or a bent specimen.

Watch it fire. A mirrored recording does **not** show up as an improper matrix — indexing
builds right-handed triads and so always returns a proper rotation. It mirrors the in-plane
axes *and* reverses the derived pattern normal, which together are a $180^\circ$ rotation
about the pattern $x$ axis: perfectly proper, and invisible to any determinant check.

In [ ]:
from pytex.tem.stage import rotation_x

mirrored = []
for indices in ([0, 0, 1], [0, 1, 1], [-1, 1, 3]):
    z = ZoneAxis(indices, phase=nickel)
    p = position_of(truth.as_matrix(), z)
    mirrored.append(
        IndexedPatternObservation(
            rotation_x(np.pi) @ indexing_would_report(truth.as_matrix(), p, TRUE_PHI_D),
            ZoneAxis([-v for v in indices], phase=nickel),  # the flipped sense indexing reports
            p,
        )
    )

bad = orientation_from_indexed_patterns(mirrored, holder)
print(f"determinants of the pattern matrices : "
      f"{[round(float(np.linalg.det(o.matrix)), 6) for o in mirrored]}  <- all proper")
print(f"beam-axis deviation                  : {bad.beam_deviation_deg:.2f} deg")
print(f"consistent                           : {bad.is_consistent}")
print()
try:
    bad.as_calibration()
except ValueError as error:
    print("as_calibration refused:", str(error)[:200], "...")

Every pattern matrix is a proper rotation, so a determinant check sees nothing wrong — and the
beam-axis deviation catches it anyway. The fit also refuses to hand back a calibration it
knows the data contradict, because propagating a contradicted constant is worse than having
none.

## 9. What to take away

- **Use two zone axes.** The two-zone reconstruction needs no diffraction rotation and no
  parity bit, and it hands you a calibration-free consistency check for free. If you have
  visited a second zone — and you almost always have — use it.
- **Ask for the orbit, not the direction.** Every symmetry equivalent gives the same
  pattern. The engine finds the cheapest reachable one, and that is frequently the
  difference between reachable and not.
- **Friedel's law is not your problem** unless your crystal is one of the ten affected
  point groups *and* your measurement is polarity-sensitive. For cubic and hexagonal metals
  it is absorbed entirely by symmetry.
- **Your diffraction rotation is your problem.** It is not in the metadata, it drifts with
  the lens history, and getting it wrong by $180^\circ$ produces a perfectly
  self-consistent answer that is exactly backwards. Two exposures fix it.
- **Break long hops into short ones.** The error from a stale calibration scales with the
  length of the move, so re-indexing at an intermediate zone converts an open-loop
  calculation into a self-correcting procedure.
- **Indexing gives you an orientation, not just a zone axis.** With the holder angles it
  becomes a full `Orientation` in Bunge angles, comparable with EBSD. With *two* patterns
  it also hands you the diffraction rotation, so the calibration problem solves itself.

### Where to go next

- {doc}`../../architecture/tem_tilt_navigation_foundation` — the derivations, the
  three-layer ambiguity analysis, the error model, and the validation strategy.
- {doc}`../../examples/generated/tem_tilt_navigation` — the executable worked examples,
  each checked against an analytic or tabulated reference value.
- {doc}`12_saed_workflows` — indexing the pattern that feeds this workflow.